# Confronto OCSE

Confronti OCSE su entrate e spesa. La prima cella richiama il codice Python del repo: se l'input non esiste, genera `source-data.json` e materializza la sezione `confronto_ocse`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from bilancio_pubblico.notebook_inputs import load_section, frame, print_input_status

SECTION_ID = "confronto_ocse"
REFRESH = False
FORCE_DOWNLOAD = False

ocse = load_section(SECTION_ID, refresh=REFRESH, force=FORCE_DOWNLOAD)
print_input_status(SECTION_ID)

In [ ]:
import matplotlib.pyplot as plt

def plot_bar(data, x, y, title, top=15):
    df = data.dropna(subset=[y]).sort_values(y, ascending=False).head(top).sort_values(y)
    ax = df.plot.barh(x=x, y=y, legend=False, figsize=(9, max(4, len(df) * 0.35)))
    ax.set_title(title)
    ax.set_xlabel(y)
    ax.set_ylabel("")
    plt.tight_layout()
    return ax

## Viste disponibili

Le viste OCSE sono separate dal confronto europeo perché fonte e perimetro sono diversi.

In [ ]:
display(frame(ocse["available_views"]))
data = ocse["data"]
data.keys()

## Pressione fiscale totale

Classifica dei paesi disponibili nel payload OCSE.

In [ ]:
tax = frame(data.get("peer_revenue_total"))
display(tax.head(20))
candidate_cols = [col for col in ["country", "paese", "code", "codice"] if col in tax.columns]
value_cols = [col for col in ["value", "valore", "percent_gdp"] if col in tax.columns]
if candidate_cols and value_cols:
    chart = tax.rename(columns={candidate_cols[0]: "country", value_cols[0]: "value"})
    plot_bar(chart, "country", "value", "Pressione fiscale totale OCSE")

## Spesa pubblica totale

Classifica dei paesi disponibili nel payload OCSE.

In [ ]:
spending = frame(data.get("peer_spending_total"))
display(spending.head(20))
candidate_cols = [col for col in ["country", "paese", "code", "codice"] if col in spending.columns]
value_cols = [col for col in ["value", "valore", "percent_gdp"] if col in spending.columns]
if candidate_cols and value_cols:
    chart = spending.rename(columns={candidate_cols[0]: "country", value_cols[0]: "value"})
    plot_bar(chart, "country", "value", "Spesa pubblica totale OCSE")

## Entrate e spese per categoria

Queste tabelle servono per capire quali categorie sono disponibili prima di costruire grafici più specifici.

In [ ]:
display(frame(data.get("revenue_categories")).head(30))
display(frame(data.get("spending_categories")).head(30))